# 加权注意力

上一章，我们用因果掩码堵上了**偷看答案**的漏洞。但是目前的因果注意力还有一个隐藏的问题：**词元身兼数职**。

回顾一下自注意力的计算过程：

$$
\text{Attention}(X) = \text{softmax}\left(\frac{X \cdot X^T}{\sqrt{d}}\right) X
$$

同一个输入 $X$ 同时扮演来三个角色：

* **提问者**：我是哪个词元？我需要什么信息？
* **回答者**：我是哪个词元？我能提供什么信息？
* **信息本身**：我是哪个词元？我实际携带的内容是什么？

同一个词元既要提问、又要回答、还是信息本身。同时充当多个角色，很难都做好。

## 查、键、值

解决的办法是实施**角色分离**，给每个词元生成 3 个不同的向量。可以想象为给同一个词元 3 个不同的面具，分别在扮演不同角色时使用：

* **Query**（查）：充当**提问者**；
* **Key**（键）：充当**回答者**；
* **Value**（值）：充当**信息本身**。

这三个向量通过对输入 $X$ 分别做**线性变换**得到：

$$
Q = X \cdot W_Q, \quad K = X \cdot W_K, \quad V = X \cdot W_V
$$

这里 $W_Q$、$W_K$、$W_V$ 是三个可训练的权重矩阵，可以通过模型训练，调整到最适合对应角色的方向。有了角色分离，自注意力的公式就变成了：

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q \cdot K^T}{\sqrt{d}}\right) V
$$

### 加权注意力

加入 $Q$、$K$、$V$ 线性投影后的注意力，称为**加权注意力**（Weighted Attention）。

In [1]:
from pathlib import Path

import requests

from src.core import *

In [2]:
np.random.seed(42)

## 数据集

### 字符编码数据集

In [3]:
class CharDataset(Dataset):

    def __init__(self, filename, batch_size=1, context_size=64, stride=None, split=0.9):
        self.filename = filename
        self.context_size = context_size
        self.stride = stride if stride is not None else context_size // 2
        self.split = split
        super().__init__(batch_size)

    def load(self):
        with open(self.filename, encoding="utf-8") as f:
            text = f.read()

        self.vocab = sorted(set(text))
        self.vocab_size = len(self.vocab)
        self.stoi = {ch: i for i, ch in enumerate(self.vocab)}
        self.itos = {i: ch for i, ch in enumerate(self.vocab)}
        self.tokens = self.encode(text)

        split = int(len(self.tokens) * self.split)
        self.train_data = self._pack(self.tokens[:split])
        self.test_data = self._pack(self.tokens[split:])

    def _pack(self, tokens):
        x, y = [], []
        for i in range(0, len(tokens) - self.context_size - 1, self.stride):
            x.append(tokens[i: i + self.context_size])
            y.append(tokens[i + 1: i + self.context_size + 1])
        return x, y

    def encode(self, symbols):
        return [self.stoi[s] for s in symbols]

    def decode(self, tokens):
        return "".join(self.itos[t] for t in tokens)

## 下三角矩阵层

In [4]:
class Tril(Layer):

    def __init__(self, value=-1e9):
        super().__init__()
        self.value = value

    def forward(self, x: Tensor):
        keep = np.tril(np.ones(x.shape[-2:]))
        p = Tensor(np.where(keep, x.data, self.value))

        def gradient_fn():
            x.grad += p.grad * keep

        return p.attach(gradient_fn, {x})

## 激活函数

### 高斯误差线性单元激活函数（GELU）

In [5]:
class GELU(Layer):

    def __init__(self):
        super().__init__()
        self.c = np.sqrt(2.0 / np.pi)

    def forward(self, x: Tensor):
        tanh = np.tanh(self.c * (x.data + 0.044715 * x.data ** 3))
        a = Tensor(0.5 * x.data * (1.0 + tanh))

        def gradient_fn():
            grad = 0.5 * (1.0 + tanh) + 0.5 * x.data * (1.0 - tanh ** 2) * self.c * (1.0 + 3.0 * 0.044715 * x.data ** 2)
            x.grad += a.grad * grad

        return a.attach(gradient_fn, parents={x})

## 损失函数

### GPT 交叉熵损失函数

In [6]:
class GPTLoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        exp = np.exp(p.data - np.max(p.data, axis=-1, keepdims=True))
        softmax = exp / np.sum(exp, axis=-1, keepdims=True)

        flat_softmax = softmax.reshape(-1, softmax.shape[-1])
        flat_y = y.data.reshape(-1).astype(np.int64)
        n = len(flat_y)
        rows = np.arange(n)

        log = np.log(np.clip(flat_softmax[rows, flat_y], 1e-10, 1))
        ce = Tensor(0 - np.sum(log) / n)

        def gradient_fn():
            flat_grad = flat_softmax.copy()
            flat_grad[rows, flat_y] -= 1
            p.grad += ce.grad * flat_grad.reshape(softmax.shape) / n

        return ce.attach(gradient_fn, parents={p})

## 模型

### GPT 嵌入层

In [7]:
class GPTEmbedding(Composite):

    def __init__(self, vocab_size, context_size, embedding_size):
        self.embedding = Embedding(vocab_size, embedding_size)
        self.positional_embedding = Embedding(context_size, embedding_size)

        super().__init__([self.embedding,
                          self.positional_embedding])

    def forward(self, x: Tensor):
        token = self.embedding(x)
        position = self.positional_embedding(Tensor(range(x.shape[1])))
        return token + position

### GPT 注意力层

我们在 GPT 注意力层引入 $Q$、$K$、$V$ 三个线性投影。前向传播步骤扩展为：

* **生成 $Q$、$K$、$V$**：通过三个线性层，分别得到查、键、值；
* **计算注意力分数矩阵**：$\text{scores} = Q \cdot K^T$
* **缩放**：乘以$1 / \sqrt{\text{embedding size}}$，防止数值过大；
* **因果掩码**：用下三角矩阵层屏蔽未来位置；
* **计算注意力权重**：用 Softmax 激活函数将每一行转换成概率分布；
* **加权求和**：注意力权重乘以 $V$，得到融合上下文的新的**注意力向量**（Attention Vector）。

In [8]:
class GPTAttention(Composite):

    def __init__(self, embedding_size):
        self.embedding_size = embedding_size

        self.query = Linear(embedding_size, embedding_size)
        self.key = Linear(embedding_size, embedding_size)
        self.value = Linear(embedding_size, embedding_size)
        self.mask = Tril()
        self.softmax = Softmax()
        self.output = Linear(embedding_size, embedding_size)

        super().__init__([self.query,
                          self.key,
                          self.value,
                          self.mask,
                          self.softmax,
                          self.output])

    def forward(self, x: Tensor):
        query = self.query(x)
        key = self.key(x)
        value = self.value(x)
        scale = Tensor(np.array(1.0 / np.sqrt(self.embedding_size)))
        scores = query @ key.transpose((0, 2, 1)) * scale
        weights = self.softmax(self.mask(scores))
        return self.output(weights @ value)

### GPT 前馈网络层

In [9]:
class GPTFeedForward(Composite):

    def __init__(self, embedding_size):
        self.input = Linear(embedding_size, embedding_size * 4)
        self.gelu = GELU()
        self.output = Linear(embedding_size * 4, embedding_size)

        super().__init__([self.input,
                          self.gelu,
                          self.output])

    def forward(self, x: Tensor):
        h = self.gelu(self.input(x))
        return self.output(h)

### GPT 转换器模块

In [10]:
class GPTTransformer(Composite):

    def __init__(self, embedding_size):
        self.attention = GPTAttention(embedding_size)
        self.feed_forward = GPTFeedForward(embedding_size)

        super().__init__([self.attention,
                          self.feed_forward])

    def forward(self, x: Tensor):
        x = self.attention(x)
        return self.feed_forward(x)

### GPT 输出层

In [11]:
class GPTOutput(Composite):

    def __init__(self, embedding_size, vocab_size):
        self.output = Linear(embedding_size, vocab_size)

        super().__init__([self.output])

    def forward(self, x: Tensor):
        return self.output(x)

### GPT 模型

In [12]:
class GPT(Composite):

    def __init__(self, vocab_size, context_size, embedding_size, blocks):
        self.embedding = GPTEmbedding(vocab_size, context_size, embedding_size)
        self.transformers = [GPTTransformer(embedding_size) for _ in range(blocks)]
        self.output = GPTOutput(embedding_size, vocab_size)

        super().__init__([self.embedding] + self.transformers + [self.output])

    def forward(self, x: Tensor):
        x = self.embedding(x)
        for layer in self.transformers:
            x = layer(x)
        return self.output(x)

## GPT 训练器

In [13]:
class GPTTrainer(Trainer):

    def train(self, dataset, epochs, scheduler=None):
        dataset.train()
        self.model.train()

        steps = 0
        for epoch in range(epochs):
            for i in range(len(dataset)):
                if scheduler is not None:
                    self.optimizer.lr = scheduler.step(steps)

                feature, label = dataset[i]

                self.optimizer.zero_grad()
                prediction = self.model(feature)
                loss = self.loss_fn(prediction, label)
                loss.backward()
                self.optimizer.clip_grad_norm()
                self.optimizer.step()
                steps += 1

    def test(self, dataset):
        dataset.eval()
        self.model.eval()

        predictions = []
        total_loss = 0.0
        with Tensor.no_grad():
            for i in range(len(dataset)):
                feature, label = dataset[i]
                prediction = self.model(feature)
                loss = self.loss_fn(prediction, label)
                predictions.append(prediction)
                total_loss += float(loss.data)

        dataset.train()
        self.model.train()
        return predictions, total_loss / len(dataset)

    def generate(self, dataset, prompt, steps=256):
        self.model.eval()
        tokens = dataset.encode(prompt)

        with Tensor.no_grad():
            for _ in range(steps):
                feature = Tensor([tokens[-dataset.context_size:]])
                logits = self.model(feature)

                last_logits = logits.data[0, -1]
                exp = np.exp(last_logits - np.max(last_logits))
                probs = exp / np.sum(exp)
                token = np.random.choice(len(probs), p=probs)
                tokens.append(token)

        return dataset.decode(tokens)

## 超参数

### 学习率

In [14]:
LEARNING_RATE = 0.001

### 批大小

In [15]:
BATCH_SIZE = 4

### 轮数

In [16]:
EPOCHS = 2

### 上下文尺寸

In [17]:
CONTEXT_SIZE = 32

### 嵌入尺寸

In [18]:
EMBEDDING_SIZE = 64

### 转换器模块数

In [19]:
BLOCKS = 2

## 数据准备

In [20]:
DATA_FILE = "../../../tinyshakespeare.txt"

file = Path(DATA_FILE)
if not file.exists():
    file.parent.mkdir(parents=True, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    response = requests.get(url)
    response.raise_for_status()
    file.write_text(response.text)

## 建模

In [21]:
dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
layer = GPT(dataset.vocab_size, CONTEXT_SIZE, EMBEDDING_SIZE, BLOCKS)
loss_fn = GPTLoss()
optimizer = AdamWOptimizer(layer.parameters, lr=LEARNING_RATE)
trainer = GPTTrainer(layer, loss_fn, optimizer)

## 训练

In [22]:
scheduler = WarmupCosineScheduler(LEARNING_RATE, EPOCHS * len(dataset), 100, LEARNING_RATE / 10)
trainer.train(dataset, EPOCHS, scheduler)

## 评估

In [23]:
prediction, loss = trainer.test(dataset)
print(f'prediction:\t{len(prediction)} steps, each {prediction[0].data.shape}\nloss:\t{loss}')

prediction:	1742 steps, each (4, 32, 65)
loss:	0.21117208628242826


## 推理

In [24]:
print(trainer.generate(dataset, prompt="ROMEO:"))

ROMEO:
Where as his and andsely,
That fleat; bedit just is it far.

TRAINCLANIA:
I see heartune:
Nopers
so at a to is an he the lady nothus
chortemort on gras band lifighten rettio su disch we man tonerraly, in my igne, leen her me. Whose nittlying:
And were;
ag


本章，我们给自注意力机制加上了**角色分离**机制：用 $Q$、$K$、$V$ 三个线性投影，让词元在**提问**、**回答**和**传递信息**时各司其职。

目前的自注意力还有一个局限：所有词元只用同一套 $Q$、$K$、$V$。就像一个人总是只用一种视角看问题，难免片面。比如**苹果**这个词，在**吃苹果**和**苹果手机**中的含义是完全不同，但目前的自注意力只会用同一种方式去理解。

下一章，我们将引入多头机制，让模型学会从不同角度理解上下文。

## 课后练习

尝试把 $Q$、$K$、$V$ 三个线性投影中的某一个去掉，用 $X$ 直接代替。观察一下训练效果有什么变化？哪个投影的影响最大？